# 00 — Setup and sanity check

Run this first in any new Colab session to confirm the data loaded correctly before doing real work in another notebook. Every notebook in this repo starts with the same bootstrap cell below.

In [ ]:
import os
import sys

from google.colab import userdata

token = userdata.get("GH_PAT")  # set once via the key icon in Colab's left sidebar

BRANCH = "issue-1-meds-reader-wrapper"  # change this back to "main" once PR #21 is merged

REPO_DIR = "ehrshot-patient-graphs"
if os.path.exists(REPO_DIR):
    # Re-running without a fresh VM (no restart) — make sure we're actually
    # on BRANCH's latest commit instead of silently keeping a stale clone.
    os.chdir(REPO_DIR)
    !git fetch origin
    !git checkout {BRANCH}
    !git reset --hard origin/{BRANCH}
else:
    !git clone --branch {BRANCH} https://{token}@github.com/patient-graph-lab/ehrshot-patient-graphs.git
    os.chdir(REPO_DIR)

!pip install -e ".[dev,data]"

# pip install -e . registers the package via a .pth file, which Python only
# reads at interpreter startup. Reinstalling into an already-running kernel
# (e.g. rerunning this cell without restarting the runtime) succeeds at the
# filesystem level -- "Successfully installed ehrgraph-0.1.0" -- but
# "import ehrgraph" still fails until the kernel restarts, because the
# running interpreter never re-scans for new sys.path entries. Adding src/
# directly sidesteps that: takes effect immediately, no restart needed,
# whether this is a fresh VM or a rerun.
src_path = os.path.join(os.getcwd(), "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

## STOP — manual step before running the next cell

The next cell reads the dataset from Google Drive — measured to be ~100x faster than direct upload on a fast connection (Drive's uploader is a real resumable HTTP upload; Colab's direct-upload widget is a slow base64 relay regardless of your bandwidth). It does **not** upload anything for you — that happens in a browser, done by you, before you run the cell below:

1. Go to **drive.google.com**
2. In **My Drive**, create a folder named exactly `ehrshot_transfer`
3. Upload all 4 files into it: `EHRSHOT_ASSETS.zip`, `EHRSHOT_MEDS.zip`, `meds_omop_ehrshot.tar.gz`, `meds_reader_omop_ehrshot.tar.gz`
4. Wait until Drive's own UI shows the upload **fully complete**, not just started
5. Only then run the next cell

**The files are *not* auto-deleted from Drive right now** (team decision, Sep 20 — see `DATA_ACCESS.md`): kept as a standing copy while a reply from Stanford on the DUA question is pending, so this one-time upload is the last time you should need to do it — future sessions can skip straight to mounting Drive and copying, no re-upload. This is a deliberately bigger exposure than the brief-pass-through version that was reviewed against the actual license text; revisit once Stanford replies.

*(No local file splitting needed for this path — `scripts/dismantle_data.py` stays available as a zero-exposure fallback if you ever want it instead, just much slower on a real connection; see its docstring.)*

In [ ]:
from google.colab import drive

from scripts.colab_setup import ensure_ehrshot_data

drive.mount("/content/drive")

DRIVE_TRANSFER_DIR = "/content/drive/MyDrive/ehrshot_transfer"

data_root = ensure_ehrshot_data(
    checksums_path="data/checksums.txt",
    source_dir=DRIVE_TRANSFER_DIR,
)

In [ ]:
# NOT run automatically right now (team decision, Sep 20) — see DATA_ACCESS.md.
# The Drive copy is being kept in place deliberately while a reply from Stanford
# is pending, so future sessions can skip re-uploading and just copy Drive -> VM.
# Once that's resolved (either direction), run this manually:
#
# from scripts.colab_setup import delete_from_drive
# delete_from_drive(DRIVE_TRANSFER_DIR)

In [ ]:
# Sanity check: confirm the extracted assets look right before doing anything else

assets_dir = data_root / "EHRSHOT_ASSETS"
for sub in ["benchmark", "features", "models", "results", "splits", "data"]:
    p = assets_dir / sub
    print(f"{sub:12s} exists={p.exists()}")

In [ ]:
import pandas as pd

labels = pd.read_csv(assets_dir / "benchmark" / "all_labels.csv")
print(labels.shape)
labels.head()

In [ ]:
import ehrgraph

print("ehrgraph package imports cleanly:", ehrgraph.__name__)

## Verify meds_reader against the real data (issue #1)

This is the one check that can't be done outside Colab — it needs the real, extracted MEDS reader database. Run the cell below and confirm it prints real events with no errors.

In [ ]:
from ehrgraph.data.loader import open_database

db_path = data_root / "meds_reader_omop_ehrshot"
db = open_database(str(db_path))

first_subject_id = next(iter(db))
subject = db[first_subject_id]
print(f"subject_id={subject.subject_id}")
for i, event in enumerate(subject.events):
    if i >= 5:
        break
    print(f"  {event.time}  {event.code}")